# LEVELS Benchmark — Prompt-Variant Sweep

Every Level 1 and Level 2 task above also has 5 hand-authored alternate phrasings
(`task.prompt_variants`) of the *exact same underlying ask* — the ground truth, expected
tools, and scoring rubric are identical across styles; only the wording/register changes:

- **novice** — plain-language, non-technical phrasing a first-time user might type
- **expert** — technical/jargon-heavy phrasing using DSI field names and operation terms
- **terse** — minimal, keyword-only phrasing
- **verbose** — over-explained, context-heavy phrasing with extra framing
- **casual** — informal, lowercase, abbreviation-heavy phrasing

`expand_prompt_variants()` turns each base task into 6 runs (`standard` + the 5 styles
above), and `run_prompt_variant_sweep()` executes every one of those **in full isolation**
(a fresh `DSIExplorer` per run, regardless of level) so a later-run style never benefits
from conversational state left over by an earlier-run style of the *same* task — otherwise
a model could "pass" a harder phrasing for free just because an earlier easier phrasing
already loaded the right database in a shared session.

**Cost/time note:** this expands 13 Level 1 + 11 Level 2 = 24 base tasks × 6 phrasings ×
`len(MODELS_TO_RUN)` models into **144 × n_models agent runs** — substantially more calls
than any other section in this notebook. Consider trying it on a small subset first (e.g.
`PROMPT_VARIANT_BASE_TASKS[:3]`) before running the full sweep, and/or temporarily reducing
`MODELS_TO_RUN` to one model.

Shared harness code (scoring, parallel runner, reports) lives in `levels_bench/`, and the task
definitions in `levels_bench/tasks/`. Each (model, task) pair runs as an independent job in a
pool of worker processes, so `max_workers` runs proceed at once. Every finished result is
appended to `levels_benchmark_results/<run_id>_<label>.partial.jsonl` as it completes. If a run
is interrupted, call `run_benchmark(..., resume=True)` to run only what's missing.

## Setup

In [1]:
# Run once per environment (not needed on every run):
# %pip install -q -r requirements.txt
# %pip install -q --no-build-isolation -e ../..

In [2]:
from levels_bench import expand_prompt_variants, init, run_benchmark, flatten, results_to_dataframe, load_results, reports, debug_task
from levels_bench.tasks import LEVEL1_TASKS, LEVEL2_TASKS, get_task

cfg = init(
    # models_to_run=["aws-gov.gpt-5.4"],   # default: every model in levels_bench.config.POSSIBLE_MODELS
    # judge_models=[...],                   # default: same list; judged scores are averaged across them
    max_workers=4,                          # concurrent agent runs; raise if the portal's rate limit allows
    task_timeout_sec=180,                   # per agent turn
    # skip_network=True,                    # skip tasks that download from oceans11.lanl.gov
)

Working directory: /Users/hhn/lanl/dsi/tools/ai_search
Run id: 20261022_063725 | models: ['aws-gov.gpt-5.4', 'anthropic.claude-sonnet-4-5-20250929-v1:0'] | judges: ['aws-gov.gpt-5.4', 'anthropic.claude-sonnet-4-5-20250929-v1:0']
Workers: 4 | turn timeout: 180s | skip_network: False


In [3]:
PROMPT_VARIANT_BASE_TASKS = list(LEVEL2_TASKS)   # add LEVEL1_TASKS for the full sweep; try [:3] first
pv_tasks = expand_prompt_variants(PROMPT_VARIANT_BASE_TASKS)
print(f"{len(PROMPT_VARIANT_BASE_TASKS)} base tasks -> {len(pv_tasks)} phrasings x "
      f"{len(cfg.models_to_run)} models = {len(pv_tasks) * len(cfg.models_to_run)} runs")

12 base tasks -> 72 phrasings x 2 models = 144 runs


## Run

In [4]:
pv_results = flatten(run_benchmark(pv_tasks, "prompt_variants"))
# pv_results = load_results("prompt_variants")
pv_df = results_to_dataframe(pv_results)
pv_df

prompt_variants: 72 tasks x 2 models = 144 runs (144 to run, 4 workers)
[  1/144] PASS    aws-gov.gpt-5.4 | L2_search_download_summarize__novice  score=0.77  21s   (elapsed 0m23s, ~54m59s left)
[  2/144] PASS    aws-gov.gpt-5.4 | L2_search_download_summarize  score=0.90  22s   (elapsed 0m23s, ~28m12s left)
[  3/144] PASS    anthropic.claude-sonnet-4-5-20250929-v1:0 | L2_search_download_summarize__novice  score=0.85  35s   (elapsed 0m36s, ~28m44s left)
[  4/144] PASS    aws-gov.gpt-5.4 | L2_search_download_summarize__expert  score=0.91  16s   (elapsed 0m38s, ~22m40s left)
[  5/144] PASS    anthropic.claude-sonnet-4-5-20250929-v1:0 | L2_search_download_summarize  score=0.83  47s   (elapsed 0m48s, ~22m40s left)
[  6/144] PASS    aws-gov.gpt-5.4 | L2_search_download_summarize__terse  score=0.92  15s   (elapsed 0m51s, ~19m54s left)
[  7/144] PASS    anthropic.claude-sonnet-4-5-20250929-v1:0 | L2_search_download_summarize__expert  score=0.88  36s   (elapsed 0m59s, ~19m26s left)
[  8/144] PAS

,model,task_id,query,prompt_style,base_task_id,level,category,status,passed,score,...,expects_abstention,hallucination_detected,checkpoint_opportunity_count,high_stakes_action_count,hedge_language_count,failure_category,judge_process_quality_score,judge_process_quality_reasoning,judge_efficiency_score,judge_efficiency_reasoning
0,aws-gov.gpt-5.4,L2_search_download_summarize,Find the NIF simulation dataset in the Ocean 1...,standard,L2_search_download_summarize,2,Analysis & Summarization,PASS,True,0.900,...,False,False,1,1,0,None,0.800,[aws-gov.gpt-5.4] The agent correctly found th...,0.470,[aws-gov.gpt-5.4] The agent completed the task...
1,aws-gov.gpt-5.4,L2_search_download_summarize__novice,Can you find the NIF dataset in the Ocean 11 c...,novice,L2_search_download_summarize,2,Analysis & Summarization,PASS,True,0.765,...,False,False,1,1,0,None,0.530,[aws-gov.gpt-5.4] The agent sensibly found the...,0.515,[aws-gov.gpt-5.4] The agent completed the task...
2,aws-gov.gpt-5.4,L2_search_download_summarize__expert,Resolve the NIF dataset record in the Ocean 11...,expert,L2_search_download_summarize,2,Analysis & Summarization,PASS,True,0.907,...,False,False,1,1,0,None,0.815,[aws-gov.gpt-5.4] The workflow correctly found...,0.430,[aws-gov.gpt-5.4] The agent completed the task...
3,aws-gov.gpt-5.4,L2_search_download_summarize__terse,"find NIF dataset, download nifcylxyz_hdf5_plt_...",terse,L2_search_download_summarize,2,Analysis & Summarization,PASS,True,0.915,...,False,False,1,1,0,None,0.830,[aws-gov.gpt-5.4] The workflow efficiently fou...,0.530,[aws-gov.gpt-5.4] The agent completed the task...
4,aws-gov.gpt-5.4,L2_search_download_summarize__verbose,As part of a multi-step data provenance exerci...,verbose,L2_search_download_summarize,2,Analysis & Summarization,PASS,True,0.880,...,False,False,1,1,0,None,0.760,[aws-gov.gpt-5.4] The workflow correctly found...,0.415,[aws-gov.gpt-5.4] The agent completed the task...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
139,anthropic.claude-sonnet-4-5-20250929-v1:0,L2_most_recent_drilldown__novice,Which dataset in Ocean 11 was updated most rec...,novice,L2_most_recent_drilldown,2,Data Discovery,PASS,True,1.000,...,False,False,0,0,0,None,1.000,"[aws-gov.gpt-5.4] The agent used a direct, eff...",1.000,[aws-gov.gpt-5.4] The agent matched the refere...
140,anthropic.claude-sonnet-4-5-20250929-v1:0,L2_most_recent_drilldown__expert,Identify the catalog record with the most rece...,expert,L2_most_recent_drilldown,2,Data Discovery,PASS,True,1.000,...,False,False,0,0,0,None,1.000,"[aws-gov.gpt-5.4] The agent used a direct, eff...",1.000,[aws-gov.gpt-5.4] The agent matched the refere...
141,anthropic.claude-sonnet-4-5-20250929-v1:0,L2_most_recent_drilldown__terse,most recently updated oceans11 dataset? has li...,terse,L2_most_recent_drilldown,2,Data Discovery,PASS,True,0.998,...,False,False,0,0,0,None,0.995,"[aws-gov.gpt-5.4] The agent used a direct, eff...",0.975,[aws-gov.gpt-5.4] The agent used a single dire...
142,anthropic.claude-sonnet-4-5-20250929-v1:0,L2_most_recent_drilldown__verbose,I'm trying to figure out which dataset in the ...,verbose,L2_most_recent_drilldown,2,Data Discovery,PASS,True,1.000,...,False,False,0,0,0,None,1.000,[aws-gov.gpt-5.4] The agent used a single effi...,1.000,[aws-gov.gpt-5.4] The agent matched the refere...


## Results

In [5]:
reports.prompt_styles(pv_df)

### Pass rate / avg score by prompt style (all tasks and models)

,pass_rate,avg_score,n
prompt_style,,,
standard,0.833,0.872,24
novice,0.833,0.849,24
expert,0.958,0.918,24
terse,0.875,0.850,24
verbose,0.958,0.929,24
casual,0.917,0.901,24


### Pass rate by base task and prompt style

prompt_style,standard,novice,expert,terse,verbose,casual
base_task_id,,,,,,
L2_cross_db_record_compare,1.0,0.5,1.0,1.0,1.0,1.0
L2_dsi_database_link_audit,1.0,1.0,1.0,1.0,1.0,1.0
L2_most_recent_drilldown,1.0,1.0,1.0,1.0,1.0,1.0
L2_nif_dens_max_extremum,1.0,1.0,1.0,1.0,1.0,1.0
L2_nif_num_arrays_tie,1.0,1.0,1.0,0.5,1.0,1.0
L2_physics_contact_report,0.5,0.5,1.0,0.5,0.5,0.5
L2_poisson_args_filename_facts,1.0,1.0,1.0,1.0,1.0,1.0
L2_poisson_field_completeness,1.0,0.5,1.0,1.0,1.0,1.0
L2_poisson_glossary_audit_1,1.0,1.0,1.0,1.0,1.0,1.0


### Phrasing sensitivity (spread in pass rate across styles)

0 = same outcome regardless of wording; higher = more sensitive to how the task was asked.

,base_task_id,model,pass_rate_spread
23,L2_simulation_tag_contacts,aws-gov.gpt-5.4,1.0
14,L2_poisson_field_completeness,anthropic.claude-sonnet-4-5-20250929-v1:0,1.0
19,L2_poisson_glossary_audit_2,aws-gov.gpt-5.4,1.0
18,L2_poisson_glossary_audit_2,anthropic.claude-sonnet-4-5-20250929-v1:0,1.0
9,L2_nif_num_arrays_tie,aws-gov.gpt-5.4,1.0
11,L2_physics_contact_report,aws-gov.gpt-5.4,1.0
1,L2_cross_db_record_compare,aws-gov.gpt-5.4,1.0
22,L2_simulation_tag_contacts,anthropic.claude-sonnet-4-5-20250929-v1:0,0.0
21,L2_search_download_summarize,aws-gov.gpt-5.4,0.0
20,L2_search_download_summarize,anthropic.claude-sonnet-4-5-20250929-v1:0,0.0


### Mean sensitivity by model

model
anthropic.claude-sonnet-4-5-20250929-v1:0    0.167
aws-gov.gpt-5.4                              0.417
Name: pass_rate_spread, dtype: float64

In [6]:
reports.failures(pv_df)

### Non-passing runs (15)

,model,task_id,status,score,notes,failure_category
6,aws-gov.gpt-5.4,L2_physics_contact_report,FAIL,0.472,Expected count 5 not found in tool results/response,MODEL_REASONING
7,aws-gov.gpt-5.4,L2_physics_contact_report__novice,FAIL,0.443,Expected count 5 not found in tool results/response,MODEL_REASONING
9,aws-gov.gpt-5.4,L2_physics_contact_report__terse,FAIL,0.490,Expected count 5 not found in tool results/response,MODEL_REASONING
10,aws-gov.gpt-5.4,L2_physics_contact_report__verbose,FAIL,0.285,Expected count 5 not found in tool results/response,MODEL_REASONING
11,aws-gov.gpt-5.4,L2_physics_contact_report__casual,FAIL,0.482,Expected count 5 not found in tool results/response,MODEL_REASONING
19,aws-gov.gpt-5.4,L2_cross_db_record_compare__novice,FAIL,0.070,Expected count 19 not found in tool results/response,MODEL_REASONING
42,aws-gov.gpt-5.4,L2_poisson_glossary_audit_2,FAIL,0.480,Missing required tools for this workflow: ['python_repl_tool'] (called: ['load_dsi_tool']),TOOL_FRAMEWORK
48,aws-gov.gpt-5.4,L2_simulation_tag_contacts,FAIL,0.497,Expected count 5 not found in tool results/response,UNKNOWN
49,aws-gov.gpt-5.4,L2_simulation_tag_contacts__novice,FAIL,0.497,Expected count 5 not found in tool results/response,MODEL_REASONING
50,aws-gov.gpt-5.4,L2_simulation_tag_contacts__expert,FAIL,0.500,Expected count 5 not found in tool results/response,MODEL_REASONING


In [7]:
reports.timing_breakdown(pv_results)

### Where does the time go?

#### Top-level phases (real wall-clock, no double counting)

,total_sec,mean_sec,count
phase_group,,,
step,1656.312,9.859,168
score,903.308,1.568,576


#### By model x top-level phase group (total seconds)

phase_group,score,step
model,,
anthropic.claude-sonnet-4-5-20250929-v1:0,439.86,1229.67
aws-gov.gpt-5.4,463.45,426.64


#### Nested inside steps (already counted above)

,total_sec,mean_sec,count
phase,,,
llm_call,1638.329,2.431,674
tool:download_file_tool,10.453,0.255,41
tool:arxiv_search_tool,3.556,3.556,1
tool:query_dsi_tool,1.638,0.006,294
tool:load_dsi_tool,0.541,0.005,112
tool:wikipedia_search_tool,0.446,0.446,1
tool:web_search_tool,0.316,0.316,1
tool:python_repl_tool,0.195,0.002,94


## Inspect / debug

Transcripts are recorded for every run. `debug_task` reruns one task in this process with live output.

In [8]:
# reports.transcript(pv_results, "L2_physics_contact_report__terse")
# tr = debug_task(get_task("L2_physics_contact_report__terse"), model="anthropic.claude-sonnet-4-5-20250929-v1:0")